### This Jupyter Notebook preprocesses and cleans the input seafood data. 
### It goes through the following steps : 
#### 0. Checks for input files
#### 1. (Preprocess) Extracts and normalizes relevant HS Codes (03,1604,1605)
#### 2. (Preprocess) Retains only relevant columns
#### 3. (Preprocess) Combines individual month files into one dataset
#### 4. (Preprocess) Standardizes consignee and shipper names
#### 5. (Cleaning) Condenses rows based on reference columns
#### 6. (Cleaning) Clusters rows based on clustering logic

#### 

#### 0. Checks for files in input folder - us_imports_2015

In [33]:
from pathlib import Path
import re

# Point to your input folder
input_folder = Path.cwd().parent / "input" / "us_imports_2015"
print("Using folder:", input_folder)
print("Exists?", input_folder.exists(), "Is dir?", input_folder.is_dir())

# Regex to extract the first number in a filename
def extract_number(path: Path):
    match = re.search(r"\d+", path.stem)  # look at filename without extension
    return int(match.group()) if match else float("inf")

# Collect and sort CSV files by number in filename
csv_paths = sorted(input_folder.rglob("*.csv"), key=extract_number)

print("CSV files found:", len(csv_paths))
for p in csv_paths:
    print("-", p.name)


Using folder: /Users/nikhitha.khasnavis/Desktop/DataCleanse/input/us_imports_2015
Exists? True Is dir? True
CSV files found: 46
- panjiva_us_imports_01_2015.csv
- panjiva_us_imports_01_2015_filtered.csv
- panjiva_us_imports_01_2015_filtered_cols.csv
- panjiva_us_imports_02_2015.csv
- panjiva_us_imports_02_2015_filtered.csv
- panjiva_us_imports_02_2015_filtered_cols.csv
- panjiva_us_imports_03_2015.csv
- panjiva_us_imports_03_2015_filtered.csv
- panjiva_us_imports_03_2015_filtered_cols.csv
- panjiva_us_imports_04_2015.csv
- panjiva_us_imports_04_2015_filtered.csv
- panjiva_us_imports_04_2015_filtered_cols.csv
- panjiva_us_imports_05_2015.csv
- panjiva_us_imports_05_2015_filtered.csv
- panjiva_us_imports_05_2015_filtered_cols.csv
- panjiva_us_imports_06_2015.csv
- panjiva_us_imports_06_2015_filtered.csv
- panjiva_us_imports_06_2015_filtered_cols.csv
- panjiva_us_imports_07_2015.csv
- panjiva_us_imports_07_2015_filtered.csv
- panjiva_us_imports_07_2015_filtered_cols.csv
- panjiva_us_impor

####

#### 1. (Preprocess) Extracts and normalizes relevant HS Codes (03,1604,1605).
#### Filters rows to keep only those with HS codes starting with "03", "1604", or "1605". Normalizes the HS tokens (fixing leading zero issues, ensuring 6/8/10-digit length)

In [34]:
from __future__ import annotations
from pathlib import Path
from typing import Optional, Tuple, List
import re
import pandas as pd
import csv

# Paths 
folder = Path("/Users/nikhitha.khasnavis/Desktop/DataCleanse/input/us_imports_2015")
out_root = folder / "hs_code"
out_root.mkdir(parents=True, exist_ok=True)


# Configuration Set Up
CHUNK_ROWS = 150_000  # read in chunks for large files
HS_CANDIDATE_NAMES: List[str] = [
    "HSCode", "HS Code", "HS_Code", "HS", "HSCODE", "HTS", "HTSCode", "HTS Code",
]
DIGIT_RE = re.compile(r"\D+")         # match non-digits
NUM_IN_STEM_RE = re.compile(r"(\d+)") # extract trailing numbers from filenames


# Functions
def normalize_hs_token(token: str) -> Optional[str]:
    """
    Normalize an HS token to a 6/8/10-digit code:
      - Remove non-digit characters.
      - Prepend '0' if starts with '3' but not '03'.
      - Only accept lengths 6,8,10.
    """
    if token is None:
        return None
    digits = DIGIT_RE.sub("", str(token))
    if not digits:
        return None
    if digits.startswith("3") and not digits.startswith("03"):
        digits = "0" + digits
    if len(digits) in (5, 7, 9):
        digits = "0" + digits
    if len(digits) not in (6, 8, 10):
        return None
    return digits

def _normalize_cell_to_tokens(cell: str) -> List[str]:
    """
    Split a cell by commas, normalize each HS code token,
    deduplicate them while preserving order.
    """
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return []
    seen, out = set(), []
    for raw in str(cell).split(","):
        tok = normalize_hs_token(raw.strip())
        if tok and tok not in seen:
            seen.add(tok)
            out.append(tok)
    return out

def any_token_matches_target(cell: str) -> bool:
    """
    Return True if any normalized token starts with the HS prefixes of interest:
    '03', '1604', or '1605'.
    """
    for tok in _normalize_cell_to_tokens(cell):
        if tok.startswith(("03", "1604", "1605")):
            return True
    return False

def score_as_hs_column(series: pd.Series, sample: int = 10_000) -> float:
    s = series.dropna().astype(str).head(sample)
    if s.empty:
        return 0.0
    good = sum(1 for val in s if _normalize_cell_to_tokens(val))
    return good / len(s)

def detect_hs_column(df: pd.DataFrame, min_score: float = 0.30) -> Optional[str]:
    for name in HS_CANDIDATE_NAMES:
        if name in df.columns:
            return name
    scores = {col: score_as_hs_column(df[col]) for col in df.columns}
    if not scores:
        return None
    best_col, best_score = max(scores.items(), key=lambda kv: kv[1])
    return best_col if best_score >= min_score else None

def write_filtered_csv(src_path: Path, dest_path: Path, hs_col: str) -> Tuple[int, int]:
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    header_written = False
    kept, total = 0, 0
    for chunk in pd.read_csv(src_path, chunksize=CHUNK_ROWS, dtype=str, low_memory=False):
        total += len(chunk)
        mask = chunk[hs_col].apply(any_token_matches_target)
        sub = chunk.loc[mask].copy()
        if not sub.empty:
            sub[hs_col] = sub[hs_col].apply(lambda cell: ",".join(_normalize_cell_to_tokens(cell)))
            sub.to_csv(dest_path, mode="a", header=not header_written, index=False, quoting=csv.QUOTE_NONNUMERIC)
            header_written = True
            kept += len(sub)
    return kept, total

def key_by_number(p: Path):
    """Sort helper: arranges CSV files in numerical order"""
    nums = NUM_IN_STEM_RE.findall(p.stem)
    return (int(nums[-1]) if nums else float("inf"), p.name.lower())

csv_paths = sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower()==".csv"], key=key_by_number)
N = len(csv_paths)

grand_total = 0
grand_kept  = 0
skipped     = 0

for i, p in enumerate(csv_paths, 1):
    out_path = out_root / f"{p.stem}_filtered.csv"

    # Detect HS column using a sample
    try:
        sample_df = pd.read_csv(p, nrows=50_000, dtype=str, low_memory=False)
    except Exception:
        print(f"{i}/{N} {p.name} → kept: 0 | input: 0 | removed: 0 (read error)")
        skipped += 1
        continue

    hs_col = detect_hs_column(sample_df)
    if not hs_col:
        print(f"{i}/{N} {p.name} → kept: 0 | input: 0 | removed: 0 (no HS column)")
        skipped += 1
        continue

    kept, total = write_filtered_csv(p, out_path, hs_col)
    removed = total - kept
    grand_total += total
    grand_kept  += kept

    print(f"{i}/{N} {p.name} → kept: {kept:,} | input: {total:,} | removed: {removed:,}")

overall_removed = grand_total - grand_kept
print(f"\nAll files saved under: {out_root}")
print(f"Totals — kept: {grand_kept:,} | input: {grand_total:,} | removed: {overall_removed:,}")


# Dataframes
if csv_paths:
    first_out = out_root / f"{csv_paths[0].stem}_filtered.csv"
    if first_out.exists():
        print(f"\nPreview of first saved file: {first_out.name}")
        df_preview = pd.read_csv(first_out, dtype=str, nrows=10)
        display(df_preview)
    else:
        print("\nNo filtered file found for preview.")
else:
    print("\nNo input CSVs were processed, nothing to preview.")


1/12 panjiva_us_imports_01_2015.csv → kept: 8,639 | input: 796,825 | removed: 788,186
2/12 panjiva_us_imports_02_2015.csv → kept: 7,898 | input: 802,028 | removed: 794,130
3/12 panjiva_us_imports_03_2015.csv → kept: 9,606 | input: 991,362 | removed: 981,756
4/12 panjiva_us_imports_04_2015.csv → kept: 7,386 | input: 882,127 | removed: 874,741
5/12 panjiva_us_imports_05_2015.csv → kept: 7,732 | input: 940,006 | removed: 932,274
6/12 panjiva_us_imports_06_2015.csv → kept: 8,491 | input: 945,398 | removed: 936,907
7/12 panjiva_us_imports_07_2015.csv → kept: 8,506 | input: 954,017 | removed: 945,511
8/12 panjiva_us_imports_08_2015.csv → kept: 8,246 | input: 998,858 | removed: 990,612
9/12 panjiva_us_imports_09_2015.csv → kept: 8,116 | input: 915,263 | removed: 907,147
10/12 panjiva_us_imports_10_2015.csv → kept: 8,523 | input: 914,694 | removed: 906,171
11/12 panjiva_us_imports_11_2015.csv → kept: 8,986 | input: 876,188 | removed: 867,202
12/12 panjiva_us_imports_12_2015.csv → kept: 9,375 |

,PanjivaRecordID,BillOfLadingNumber,ArrivalDate,DataLoadDate,DataLaunchDate,ConsigneeName,ConsigneeFullAddress,ConsigneeRoute,ConsigneeCity,ConsigneeStateRegion,...,HasLCL,ContainerNumbers,HSCode,GoodsShipped,VolumeContainerTEU,ContainerMarks,DividedLCL,ContainerTypeOfService,ContainerTypes,DangerousGoods
0,106859389,OOLU2020226780,2015-01-10,2015-01-12,2015-01-14,Sealand Food Inc.,"7418 RANCO ROAD RICHMOND, VA 23228",7418 Ranco Road,Richmond,Virginia,...,NaN,OOLU6411963,030461,FROZEN TILAPIA FILLETS,2.0,NaN,N,House to House,45R1,false
1,106891645,COSU6107606760B5,2015-01-10,2015-01-13,2015-01-15,High Liner Foods Inc.,18 ELECTRONICS AV DANVERS MA 0192,1 Highliner Avenue,Portsmouth,New Hampshire,...,NaN,SZLU9678305,030429,FROZEN HADDOCK FILLETS,2.0,NaN,N,House to House,4430,false
2,106828349,HLCUCA4141065246,2015-01-10,2015-01-12,2015-01-14,The Fishin Co.,"3714 MAIN STREET MUNHALL, PA 15120",3714 Main Street,Munhall,Pennsylvania,...,NaN,HLXU8728308,030461,"FROZEN TILAPIA FILLET, 8X4LB PO 14/88T-18 BRA...",2.0,. . . . . . . . . . . .,N,House to House,4532,false
3,106803773,APLU053188531,2015-01-08,2015-01-10,2015-01-12,Grupo Almos Sa De Cv,INDUSTRIA NO. 10 COL. AZCAPOTZALCO MEXICOD.F. ...,NaN,NaN,NaN,...,NaN,CAIU5534187,030461,FROZEN TILAPIA FILLET 3-5OZ -OREOCHROMIS NILOT...,2.0,NaN,N,Container Yard,45R0,false
4,106916541,NYKS2410249680,2015-01-10,2015-01-13,2015-01-15,Bumble Bee Foods Llc,9655 GRANITE RIDGE DRIVE SAN DIEGO CA 92123,9655 Granite Ridge Drive,San Diego,California,...,NaN,NYKU2703505,160590,BUMBLE BEE BRAND BOILED WHOLE OYSTERS SHIPPERS...,1.0,NaN,N,House to House,22G1,false
5,106659965,SEYOPANY1411298,2015-01-04,2015-01-05,2015-01-07,High Liner Foods Inc.,18 ELECTRONICS AV DANVERS MA 0192,1 Highliner Avenue,Portsmouth,New Hampshire,...,NaN,ZCSU5836210,030471,FROZEN PACIFIC COD FILLETS,2.0,NaN,N,House to House,45R1,false
6,106627901,MOLU13013824846,2015-01-04,2015-01-05,2015-01-07,Beaver Street Fisheries,1741 WEST BEAVER ST JACKSONVILLE FLORIDA USA 3...,1741 West Beaver Street,Jacksonville,Florida,...,NaN,MORU1306800,030461,2514 CTN 1) FROZEN INDIVIDUALLY VACUUM PACKED...,2.0,2514 CTN 1) FROZEN INDIVIDUALLY VACUUM PACKED...,N,Container Yard,4536,false
7,106724669,SILQCNDLC0012218,2015-01-06,2015-01-07,2015-01-09,Polar Bay Foods Inc.,1750 112TH AV #C236 BELLEVUE WA 98004 USA,1750 112th Avenue Northeast,Bellevue,Washington,...,NaN,SZLU9884970,030471,IQF COD PORTIONS CUTTING FROM FILLETS INVOICE ...,2.0,NaN,N,House to House,45R0,false
8,107257533,APLU065658760,2015-01-25,2015-01-26,2015-01-28,Yellow Rlver Corp.,1251 E.VALLEY BLVD ALHAMBRA CA 91801 USA,1251 East Valley Boulevard,Alhambra,California,...,NaN,GESU9446641,030389,DRIED SHRIMP SKIN FROZEN WILD BEHEADED YELLOW ...,2.0,NaN,N,Container Yard,45R0,false
9,106992509,ITGB3220839P5282,2015-01-15,2015-01-16,2015-01-18,Port Royal Sales,95 FROEHLICH FARM BLVD WOODBURY NY 11797 USA,NaN,NaN,NaN,...,NaN,CMAU1166768,160414,CANNED CHUNK LIGHT FRIGATE TUNA (AUXIS THAZARD...,1.0,NaN,N,Pier to Pier,2200,false


####

#### 2. (Preprocess) Retains only relevant columns
#### Retain only relevant reference columns (defined in BUSINESS_KEEP_COLS) and output the files 

In [35]:
from pathlib import Path
from typing import Optional, Tuple
import re
import pandas as pd

# Paths
in_folder  = Path("/Users/nikhitha.khasnavis/Desktop/DataCleanse/input/us_imports_2015/hs_code")
out_folder = Path("/Users/nikhitha.khasnavis/Desktop/DataCleanse/input/us_imports_2015/columns")
out_folder.mkdir(parents=True, exist_ok=True)

CHUNK_ROWS = 150_000  # process in chunks to handle large files efficiently

# Columns to keep if present
BUSINESS_KEEP_COLS = [
    # Consignee
    "ConsigneeName", "ConsigneeFullAddress", "ConsigneeLocalDUNS",
    "ConsigneePanjivaID", "ConsigneeOriginalFormat",

    # Shipper
    "ShipperName", "ShipperFullAddress", "ShipperLocalDUNS",
    "ShipperPanjivaID", "ShipperOriginalFormat",

    # Weight / quantity variants
    "GrossWeightKg", "NetWeightKg", "WeightKg", "Weightkg",
    "Quantity (kg)", "Quantity_kg", "QuantityKg",

    # Value (USD) variants
    "ValueUSD", "Value_USD", "FOBUSD", "ValueOfGoodsFOBUSD", "CIFUSD",
    "ExportValue", "ValueOfGoodsUSD", "InvoiceValueUSD", "Value",
]

# Candidate HS column names
HS_CANDIDATE_NAMES = [
    "HSCode", "HS Code", "HS_Code", "HS", "HSCODE", "HTS", "HTSCode", "HTS Code",
]

# Functions
def _score_as_hs_column(series: pd.Series, sample: int = 10_000) -> float:
    s = series.dropna().astype(str).head(sample)
    if s.empty:
        return 0.0
    keep = sum(1 for val in s if re.search(r"\d", val))
    return keep / len(s)

def detect_hs_column(df: pd.DataFrame, min_score: float = 0.30) -> Optional[str]:
    for name in HS_CANDIDATE_NAMES:
        if name in df.columns:
            return name
    scores = {col: _score_as_hs_column(df[col]) for col in df.columns}
    if not scores:
        return None
    best_col, best_score = max(scores.items(), key=lambda kv: kv[1])
    return best_col if best_score >= min_score else None

# Ensure CSVs are processed in natural numeric order (01,02,...12)
num_re = re.compile(r"(\d+)")
def sort_key(p: Path):
    nums = num_re.findall(p.stem)
    return (int(nums[-1]) if nums else float("inf"), p.name.lower())

csv_paths = sorted(
    [p for p in in_folder.iterdir() if p.is_file() and p.suffix.lower() == ".csv"],
    key=sort_key
)

def write_slimmed_csv(src: Path, dest: Path) -> Tuple[int, int, int]:
    # Read only the header first
    head = pd.read_csv(src, nrows=0, dtype=str, low_memory=False)
    hs_col = detect_hs_column(head)

    # Fallback: sample rows if not found
    if not hs_col:
        sample = pd.read_csv(src, nrows=50_000, dtype=str, low_memory=False)
        hs_col = detect_hs_column(sample)
    if not hs_col:
        return (0, head.shape[1], 0)

    # Select keep columns
    keep_found = [c for c in BUSINESS_KEEP_COLS if c in head.columns]
    usecols = [hs_col] + [c for c in keep_found if c != hs_col]

    # Stream in chunks and write
    header_written = False
    rows_written = 0
    for chunk in pd.read_csv(src, usecols=usecols, chunksize=CHUNK_ROWS, dtype=str, low_memory=False):
        chunk.to_csv(dest, mode="a", header=not header_written, index=False)
        header_written = True
        rows_written += len(chunk)

    return (rows_written, head.shape[1], len(usecols))

grand_rows = 0
first_output_path = None
in_cols_final = None
out_cols_final = None

for i, p in enumerate(csv_paths, 1):
    out_path = out_folder / f"{p.stem}_cols.csv"
    rows, in_cols, out_cols = write_slimmed_csv(p, out_path)
    grand_rows += rows
    if first_output_path is None and rows > 0:
        first_output_path = out_path
        in_cols_final = in_cols
        out_cols_final = out_cols

print("\nFiles saved under:", out_folder)
print(f"Total rows across all outputs: {grand_rows:,}")
if in_cols_final is not None and out_cols_final is not None:
    print(f"Number of columns decreased from {in_cols_final} → {out_cols_final}")

if first_output_path and first_output_path.exists():
    print("\nPreview of first edited file:", first_output_path.name)
    df_preview = pd.read_csv(first_output_path, nrows=10, dtype=str, low_memory=False)
    display(df_preview)
else:
    print("\n(No non-empty outputs to preview.)")
    


Files saved under: /Users/nikhitha.khasnavis/Desktop/DataCleanse/input/us_imports_2015/columns
Total rows across all outputs: 101,504
Number of columns decreased from 77 → 13

Preview of first edited file: panjiva_us_imports_01_2015_filtered_cols.csv


,ConsigneeName,ConsigneeFullAddress,ConsigneeLocalDUNS,ConsigneePanjivaID,ConsigneeOriginalFormat,ShipperName,ShipperFullAddress,ShipperLocalDUNS,ShipperPanjivaID,ShipperOriginalFormat,WeightKg,ValueOfGoodsUSD,HSCode
0,Sealand Food Inc.,"7418 RANCO ROAD RICHMOND, VA 23228",185277584,33624817,SEALAND FOODS INC 7418 RANCO ROAD RICHMOND VA ...,Zhanjiang Shuanghu Food,NO. 2 INDUSTRIAL SOUTH ROAD BAINIPO INDUSTRIAL...,543261687,32579395,ZHANJIANG SHUANGHU FOOD CO LTD BAINIPO INDUSTR...,22000.0,99000.0,030461
1,High Liner Foods Inc.,18 ELECTRONICS AV DANVERS MA 0192,001110055,27838533,HIGH LINER FOODS INC. 1 HIGH LINER AVENUE PORT...,Qingdao Dencan Seafood Co.,LTD. SEASHORE INDUSTRIAL ZONE JIAON AN QINGDAO...,NaN,35036079,"QINGDAO DENCAN SEAFOOD CO.,LTD. SEASHORE INDUS...",25050.0,NaN,030429
2,The Fishin Co.,"3714 MAIN STREET MUNHALL, PA 15120",127951213,27833073,"THE FISHIN COMPANY, 3714 MAIN STREET PITTSBURG...","Xihe Food Co., Ltd.",NO.211 HK RD INDUSTRIAL ZO,NaN,2167447,"XIHE FOOD CO.,LTD. NO.211 HONG KONG ROAD, INDU...",23200.0,NaN,030461
3,Grupo Almos Sa De Cv,INDUSTRIA NO. 10 COL. AZCAPOTZALCO MEXICOD.F. ...,812138857,33698058,"GRUPO ALMOS S.A. DE C.V., INDUSTRIA NO. 10 COL...",Zhongshan Foodstuffs And Aquatic,113 HUAYUAN STREET ZHONGSHAN 3RD ROAD ZHONGSHA...,653990929,4069344,ZHONGSHAN FOODSTUFFS AND AQUATIC IM AND EXP.GR...,25160.0,113000.0,030461
4,Bumble Bee Foods Llc,9655 GRANITE RIDGE DRIVE SAN DIEGO CA 92123,135952609,27818233,"BUMBLE BEE FOODS, LLC 9655 GRANITE RIDGE DRIVE...",Oceanview Group Inc.,"Room 1107, No. 100, Middle Hongkong Road, Qing...",NaN,27792064,OCEANVIEW GROUP INC. NO.100 HONGKONG MIDDLE RO...,17000.0,NaN,160590
5,High Liner Foods Inc.,18 ELECTRONICS AV DANVERS MA 0192,001110055,27838533,HIGH LINER FOODS INC. 1 HIGH LINER AVENUE PORT...,"Qingdao Deli Trade Co., Ltd.",BLDG 2 NO.6 MINJIANG RD SHINAN DISTRICT QINGDAO,528740687,35552095,"QINGDAO DELUXE TRADING CO.,LTD ROOM 608,BUILDI...",24000.0,133000.0,030471
6,Beaver Street Fisheries,1741 WEST BEAVER ST JACKSONVILLE FLORIDA USA 3...,004079364,27830920,"BEAVER STREET FISHERIES, INC., 1741 WEST BEAVE...",Guangxi Nanning Baiyang Food Co.,CO. LTD. NO. 16 CHUANGXIN XI RD NEW AND HIGH-T...,NaN,5096630,"GUANGXI NANNING BAIYANG FOOD CO., L NO. 16, CH...",22917.0,103000.0,030461
7,Polar Bay Foods Inc.,1750 112TH AV #C236 BELLEVUE WA 98004 USA,014071704,27838906,POLAR BAY FOODS INC. 1750 112TH AVE NE SUITE C...,"Dalian Meihe Foodstuff Co., Ltd.",HONGTA VILLAGE YONGZHENG ST JINZHOU DISTRICT D...,529169771,2284978,"DALIAN MEIHE FOODSTUFF CO.,LTD. HONGTA VILLAGE...",23000.0,127000.0,030471
8,Yellow Rlver Corp.,1251 E.VALLEY BLVD ALHAMBRA CA 91801 USA,NaN,34512845,YELLOW RLVER CORP 1251 E.VALLEY BLVD ALHAMBRA ...,Fujian Yuehai Aquatic,COMPANY NO 1 FEILUAN INDUSTRY PARK JIAOCHENG D...,421369264,45866180,FUJIAN YUEHAI AQUATIC FOOD LIMITED COMPANY NO ...,19920.0,73200.0,030389
9,Port Royal Sales,95 FROEHLICH FARM BLVD WOODBURY NY 11797 USA,130886500,27889109,"PORT ROYAL SALES, LTD 95 FROEHLICH FARM BLVD W...",Tropical Food Mfg. (Ningbo),78 BINJIANG ZHILU XIAOGANG NINGBOCT:0574862285...,NaN,3919601,TROPICAL FOOD MANUFACTURING (NINGBO 78 BINJIAN...,19958.0,83200.0,160414


####

#### 3. (Preprocess) Combines individual month files into one dataset
#### After the first two preprocessing steps, the individual month files are combined into one final output for easier standardization

In [36]:
import pandas as pd
from pathlib import Path
import re

# Paths
base_folder  = Path("/Users/nikhitha.khasnavis/Desktop/DataCleanse/input/us_imports_2015")
input_folder = base_folder / "columns"
output_file  = base_folder / "us_import_2015_combined.csv"

csv_files = list(input_folder.glob("*.csv"))

# Functions
def numerical_sort(path: Path):
    """Extract numbers from filename for natural numeric ordering."""
    nums = re.findall(r"\d+", path.stem)
    return [int(n) for n in nums] if nums else [float("inf")]

csv_files = sorted(csv_files, key=numerical_sort)
print(f"Found {len(csv_files)} CSV files in {input_folder}")

# Read and combine 
df_list = []
for f in csv_files:
    # Read each CSV as string-typed DataFrame (safe for IDs, codes, etc.)
    df = pd.read_csv(f, dtype=str, low_memory=False)
    df_list.append(df)

# Concatenate all DataFrames into one
combined_df = pd.concat(df_list, ignore_index=True)

output_file.parent.mkdir(parents=True, exist_ok=True)
combined_df.to_csv(output_file, index=False)

print(f"\nFinal combined file saved to: {output_file}")
print(f"Total rows: {combined_df.shape[0]:,}")
print(f"Total cols: {combined_df.shape[1]:,}")

display(combined_df.head(10))


Found 12 CSV files in /Users/nikhitha.khasnavis/Desktop/DataCleanse/input/us_imports_2015/columns

Final combined file saved to: /Users/nikhitha.khasnavis/Desktop/DataCleanse/input/us_imports_2015/us_import_2015_combined.csv
Total rows: 101,504
Total cols: 13


,ConsigneeName,ConsigneeFullAddress,ConsigneeLocalDUNS,ConsigneePanjivaID,ConsigneeOriginalFormat,ShipperName,ShipperFullAddress,ShipperLocalDUNS,ShipperPanjivaID,ShipperOriginalFormat,WeightKg,ValueOfGoodsUSD,HSCode
0,Sealand Food Inc.,"7418 RANCO ROAD RICHMOND, VA 23228",185277584,33624817,SEALAND FOODS INC 7418 RANCO ROAD RICHMOND VA ...,Zhanjiang Shuanghu Food,NO. 2 INDUSTRIAL SOUTH ROAD BAINIPO INDUSTRIAL...,543261687,32579395,ZHANJIANG SHUANGHU FOOD CO LTD BAINIPO INDUSTR...,22000.0,99000.0,030461
1,High Liner Foods Inc.,18 ELECTRONICS AV DANVERS MA 0192,001110055,27838533,HIGH LINER FOODS INC. 1 HIGH LINER AVENUE PORT...,Qingdao Dencan Seafood Co.,LTD. SEASHORE INDUSTRIAL ZONE JIAON AN QINGDAO...,NaN,35036079,"QINGDAO DENCAN SEAFOOD CO.,LTD. SEASHORE INDUS...",25050.0,NaN,030429
2,The Fishin Co.,"3714 MAIN STREET MUNHALL, PA 15120",127951213,27833073,"THE FISHIN COMPANY, 3714 MAIN STREET PITTSBURG...","Xihe Food Co., Ltd.",NO.211 HK RD INDUSTRIAL ZO,NaN,2167447,"XIHE FOOD CO.,LTD. NO.211 HONG KONG ROAD, INDU...",23200.0,NaN,030461
3,Grupo Almos Sa De Cv,INDUSTRIA NO. 10 COL. AZCAPOTZALCO MEXICOD.F. ...,812138857,33698058,"GRUPO ALMOS S.A. DE C.V., INDUSTRIA NO. 10 COL...",Zhongshan Foodstuffs And Aquatic,113 HUAYUAN STREET ZHONGSHAN 3RD ROAD ZHONGSHA...,653990929,4069344,ZHONGSHAN FOODSTUFFS AND AQUATIC IM AND EXP.GR...,25160.0,113000.0,030461
4,Bumble Bee Foods Llc,9655 GRANITE RIDGE DRIVE SAN DIEGO CA 92123,135952609,27818233,"BUMBLE BEE FOODS, LLC 9655 GRANITE RIDGE DRIVE...",Oceanview Group Inc.,"Room 1107, No. 100, Middle Hongkong Road, Qing...",NaN,27792064,OCEANVIEW GROUP INC. NO.100 HONGKONG MIDDLE RO...,17000.0,NaN,160590
5,High Liner Foods Inc.,18 ELECTRONICS AV DANVERS MA 0192,001110055,27838533,HIGH LINER FOODS INC. 1 HIGH LINER AVENUE PORT...,"Qingdao Deli Trade Co., Ltd.",BLDG 2 NO.6 MINJIANG RD SHINAN DISTRICT QINGDAO,528740687,35552095,"QINGDAO DELUXE TRADING CO.,LTD ROOM 608,BUILDI...",24000.0,133000.0,030471
6,Beaver Street Fisheries,1741 WEST BEAVER ST JACKSONVILLE FLORIDA USA 3...,004079364,27830920,"BEAVER STREET FISHERIES, INC., 1741 WEST BEAVE...",Guangxi Nanning Baiyang Food Co.,CO. LTD. NO. 16 CHUANGXIN XI RD NEW AND HIGH-T...,NaN,5096630,"GUANGXI NANNING BAIYANG FOOD CO., L NO. 16, CH...",22917.0,103000.0,030461
7,Polar Bay Foods Inc.,1750 112TH AV #C236 BELLEVUE WA 98004 USA,014071704,27838906,POLAR BAY FOODS INC. 1750 112TH AVE NE SUITE C...,"Dalian Meihe Foodstuff Co., Ltd.",HONGTA VILLAGE YONGZHENG ST JINZHOU DISTRICT D...,529169771,2284978,"DALIAN MEIHE FOODSTUFF CO.,LTD. HONGTA VILLAGE...",23000.0,127000.0,030471
8,Yellow Rlver Corp.,1251 E.VALLEY BLVD ALHAMBRA CA 91801 USA,NaN,34512845,YELLOW RLVER CORP 1251 E.VALLEY BLVD ALHAMBRA ...,Fujian Yuehai Aquatic,COMPANY NO 1 FEILUAN INDUSTRY PARK JIAOCHENG D...,421369264,45866180,FUJIAN YUEHAI AQUATIC FOOD LIMITED COMPANY NO ...,19920.0,73200.0,030389
9,Port Royal Sales,95 FROEHLICH FARM BLVD WOODBURY NY 11797 USA,130886500,27889109,"PORT ROYAL SALES, LTD 95 FROEHLICH FARM BLVD W...",Tropical Food Mfg. (Ningbo),78 BINJIANG ZHILU XIAOGANG NINGBOCT:0574862285...,NaN,3919601,TROPICAL FOOD MANUFACTURING (NINGBO 78 BINJIAN...,19958.0,83200.0,160414


####

#### 4. (Preprocess) Standardizes consignee and shipper names
#### Applies normalization rules to company names, removes unwanted punctuation and unifies spacing

In [54]:
import re, unicodedata
import pandas as pd
from pathlib import Path

# Paths
base = Path("/Users/nikhitha.khasnavis/Desktop/DataCleanse/input/us_imports_2015")
src_file = base / "us_import_2015_combined.csv"
std_dir  = base / "std"
std_dir.mkdir(parents=True, exist_ok=True)

out_std_file   = std_dir / "us_import_2015_combined_std.csv"
out_audit_file = std_dir / "name_standardization_audit.csv"

assert src_file.exists(), f"Input not found: {src_file}"

# Regex replacements 
# Order matters — more specific patterns must appear first
REPLACEMENTS = [
    (re.compile(r"\bs\.?\s*de\s*r\.?\s*l\.?\s*de\s*c\.?\s*v\.?\b", re.I), "S. de R.L. de C.V."),
    # Mexican forms
    (re.compile(r"\bs\.?\s*a\.?\s*de\s*c\.?\s*v\.?\b", re.I), "S.A. de C.V."),
    (re.compile(r"\bs\.?\s*de\s*r\.?\s*l\.?\b", re.I),       "S. de R.L."),
    # English composites
    (re.compile(r"\bpty\s+limited\b", re.I),             "Pty Ltd"),
    (re.compile(r"\bco\.\s*,?\s*ltd\b", re.I),           "Company Limited"),
    (re.compile(r"\bco\s*,?\s*ltd\b", re.I),             "Company Limited"),
    (re.compile(r"\bcompany\s+limited\b", re.I),         "Company Limited"),
    # Single-word variants
    (re.compile(r"\bincorporated\b|\binc\b\.?", re.I),   "Incorporated"),
    (re.compile(r"\bcorporation\b|\bcorp\b\.?", re.I),   "Corporation"),
    (re.compile(r"\bcompany\b|\bco\b\.?", re.I),         "Company"),
    (re.compile(r"\blimited\b|\bltd\b\.?|\bltda\b\.?", re.I), "Limited"),
    # Acronyms (forced upper)
    (re.compile(r"\bplc\b", re.I), "PLC"),
    (re.compile(r"\bllc\b", re.I), "LLC"),
    (re.compile(r"\bllp\b", re.I), "LLP"),
    (re.compile(r"\bgmbh\b", re.I), "GMBH"),
    (re.compile(r"\bag\b", re.I), "AG"),
    (re.compile(r"\bbv\b", re.I), "BV"),
    (re.compile(r"\bnv\b", re.I), "NV"),
    (re.compile(r"\bsa\b", re.I), "SA"),
    (re.compile(r"\bsrl\b", re.I), "SRL"),
    (re.compile(r"\bspa\b", re.I), "SPA"),
    (re.compile(r"\bab\b", re.I), "AB"),  # keep AB uppercase
    # Collapse separated Pty + Ltd
    (re.compile(r"\bpty\b\s+\bltd\b", re.I), "Pty Ltd"),
]

# Functions
PUNCT_TO_SPACE = re.compile(r"[\,\.;:\|\(\)\[\]\{\}/\\\+\=\*\!\?\#\^\"“”‘’`~_–—\-]+")
WS_RE = re.compile(r"\s+")

def _nfkd_lower(s: str) -> str:
    """Normalize to ASCII + lowercase for uniform matching."""
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii").lower()

def _normalize_punct_ws(s: str) -> str:
    """Replace punctuation with spaces, collapse whitespace, keep '&' as 'and'."""
    s = s.replace("&", " and ")
    s = PUNCT_TO_SPACE.sub(" ", s)
    s = WS_RE.sub(" ", s).strip()
    return s

def standardize_name(raw) -> str:
    """Apply regex replacements and fix casing of tokens."""
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return ""
    # Normalize and clean punctuation/spacing
    s = _nfkd_lower(str(raw))
    s = _normalize_punct_ws(s)
    # Apply regex replacements
    for pat, repl in REPLACEMENTS:
        s = pat.sub(repl, s)
    # Token-level casing rules
    tokens = s.split()
    ALWAYS_UPPER = {"LLC","LLP","PLC","GMBH","AG","BV","NV","SA","SRL","SPA","AB"}
    ACRONYM_WITH_DOTS = {"S.A.", "R.L.", "C.V."}
    KEEP_AS_IS = {"Pty", "Ltd"}
    def nice_case(tok: str) -> str:
        if tok in ACRONYM_WITH_DOTS: return tok
        if tok.upper() in ALWAYS_UPPER: return tok.upper()
        if tok.lower() == "de": return "de"   # Spanish preposition
        if tok in KEEP_AS_IS: return tok
        return tok.title()
    return " ".join(nice_case(t) for t in tokens).strip()

df = pd.read_csv(src_file, dtype=str, low_memory=False)
fields = [c for c in ("ConsigneeName", "ShipperName") if c in df.columns]

# Save originals for comparison
originals = {c: df[c].copy() for c in fields}

# Apply standardization
for c in fields:
    df[c] = df[c].apply(standardize_name)

# Build audit dataframe (only rows that changed)
def build_diffs(col: str):
    before = originals[col].fillna("").astype(str).str.strip()
    after  = df[col].fillna("").astype(str).str.strip()
    mask = before.ne(after)
    return pd.DataFrame({"Field": col, "Before": before[mask], "After": after[mask]})

audit_df = pd.concat([build_diffs(c) for c in fields], ignore_index=True) if fields else pd.DataFrame(columns=["Field","Before","After"])

# Save results 
df.to_csv(out_std_file, index=False)
audit_df.to_csv(out_audit_file, index=False)

# Stats 
print(f"Saved standardized file → {out_std_file}")
print(f"Saved audit file        → {out_audit_file}")
print(f"Rows: {len(df):,} | Cols: {df.shape[1]}")
print(f"Changes recorded: {len(audit_df):,}")

for c in fields:
    old_unique = originals[c].nunique(dropna=True)
    new_unique = df[c].nunique(dropna=True)
    changed_rows = (originals[c] != df[c]).sum()
    print(f"\n{c}: {old_unique:,} unique old names → {new_unique:,} unique new names | Rows changed: {changed_rows:,}")

print("\nPreview audit:")
display(audit_df.head(10))

print("\nPreview standardized dataset:")
display(df.head(10))


Saved standardized file → /Users/nikhitha.khasnavis/Desktop/DataCleanse/input/us_imports_2015/std/us_import_2015_combined_std.csv
Saved audit file        → /Users/nikhitha.khasnavis/Desktop/DataCleanse/input/us_imports_2015/std/name_standardization_audit.csv
Rows: 101,504 | Cols: 13
Changes recorded: 117,998

ConsigneeName: 5,489 unique old names → 5,447 unique new names | Rows changed: 68,436

ShipperName: 4,867 unique old names → 4,799 unique new names | Rows changed: 68,619

Preview audit:


,Field,Before,After
0,ConsigneeName,Sealand Food Inc.,Sealand Food Incorporated
1,ConsigneeName,High Liner Foods Inc.,High Liner Foods Incorporated
2,ConsigneeName,The Fishin Co.,The Fishin Company
3,ConsigneeName,Grupo Almos Sa De Cv,Grupo Almos S.A. de C.V.
4,ConsigneeName,Bumble Bee Foods Llc,Bumble Bee Foods LLC
5,ConsigneeName,High Liner Foods Inc.,High Liner Foods Incorporated
6,ConsigneeName,Polar Bay Foods Inc.,Polar Bay Foods Incorporated
7,ConsigneeName,Yellow Rlver Corp.,Yellow Rlver Corporation
8,ConsigneeName,High Liner Foods Inc.,High Liner Foods Incorporated
9,ConsigneeName,L & Y International Co.,L And Y International Company



Preview standardized dataset:


,ConsigneeName,ConsigneeFullAddress,ConsigneeLocalDUNS,ConsigneePanjivaID,ConsigneeOriginalFormat,ShipperName,ShipperFullAddress,ShipperLocalDUNS,ShipperPanjivaID,ShipperOriginalFormat,WeightKg,ValueOfGoodsUSD,HSCode
0,Sealand Food Incorporated,"7418 RANCO ROAD RICHMOND, VA 23228",185277584,33624817,SEALAND FOODS INC 7418 RANCO ROAD RICHMOND VA ...,Zhanjiang Shuanghu Food,NO. 2 INDUSTRIAL SOUTH ROAD BAINIPO INDUSTRIAL...,543261687,32579395,ZHANJIANG SHUANGHU FOOD CO LTD BAINIPO INDUSTR...,22000.0,99000.0,030461
1,High Liner Foods Incorporated,18 ELECTRONICS AV DANVERS MA 0192,001110055,27838533,HIGH LINER FOODS INC. 1 HIGH LINER AVENUE PORT...,Qingdao Dencan Seafood Company,LTD. SEASHORE INDUSTRIAL ZONE JIAON AN QINGDAO...,NaN,35036079,"QINGDAO DENCAN SEAFOOD CO.,LTD. SEASHORE INDUS...",25050.0,NaN,030429
2,The Fishin Company,"3714 MAIN STREET MUNHALL, PA 15120",127951213,27833073,"THE FISHIN COMPANY, 3714 MAIN STREET PITTSBURG...",Xihe Food Company Limited,NO.211 HK RD INDUSTRIAL ZO,NaN,2167447,"XIHE FOOD CO.,LTD. NO.211 HONG KONG ROAD, INDU...",23200.0,NaN,030461
3,Grupo Almos S.A. de C.V.,INDUSTRIA NO. 10 COL. AZCAPOTZALCO MEXICOD.F. ...,812138857,33698058,"GRUPO ALMOS S.A. DE C.V., INDUSTRIA NO. 10 COL...",Zhongshan Foodstuffs And Aquatic,113 HUAYUAN STREET ZHONGSHAN 3RD ROAD ZHONGSHA...,653990929,4069344,ZHONGSHAN FOODSTUFFS AND AQUATIC IM AND EXP.GR...,25160.0,113000.0,030461
4,Bumble Bee Foods LLC,9655 GRANITE RIDGE DRIVE SAN DIEGO CA 92123,135952609,27818233,"BUMBLE BEE FOODS, LLC 9655 GRANITE RIDGE DRIVE...",Oceanview Group Incorporated,"Room 1107, No. 100, Middle Hongkong Road, Qing...",NaN,27792064,OCEANVIEW GROUP INC. NO.100 HONGKONG MIDDLE RO...,17000.0,NaN,160590
5,High Liner Foods Incorporated,18 ELECTRONICS AV DANVERS MA 0192,001110055,27838533,HIGH LINER FOODS INC. 1 HIGH LINER AVENUE PORT...,Qingdao Deli Trade Company Limited,BLDG 2 NO.6 MINJIANG RD SHINAN DISTRICT QINGDAO,528740687,35552095,"QINGDAO DELUXE TRADING CO.,LTD ROOM 608,BUILDI...",24000.0,133000.0,030471
6,Beaver Street Fisheries,1741 WEST BEAVER ST JACKSONVILLE FLORIDA USA 3...,004079364,27830920,"BEAVER STREET FISHERIES, INC., 1741 WEST BEAVE...",Guangxi Nanning Baiyang Food Company,CO. LTD. NO. 16 CHUANGXIN XI RD NEW AND HIGH-T...,NaN,5096630,"GUANGXI NANNING BAIYANG FOOD CO., L NO. 16, CH...",22917.0,103000.0,030461
7,Polar Bay Foods Incorporated,1750 112TH AV #C236 BELLEVUE WA 98004 USA,014071704,27838906,POLAR BAY FOODS INC. 1750 112TH AVE NE SUITE C...,Dalian Meihe Foodstuff Company Limited,HONGTA VILLAGE YONGZHENG ST JINZHOU DISTRICT D...,529169771,2284978,"DALIAN MEIHE FOODSTUFF CO.,LTD. HONGTA VILLAGE...",23000.0,127000.0,030471
8,Yellow Rlver Corporation,1251 E.VALLEY BLVD ALHAMBRA CA 91801 USA,NaN,34512845,YELLOW RLVER CORP 1251 E.VALLEY BLVD ALHAMBRA ...,Fujian Yuehai Aquatic,COMPANY NO 1 FEILUAN INDUSTRY PARK JIAOCHENG D...,421369264,45866180,FUJIAN YUEHAI AQUATIC FOOD LIMITED COMPANY NO ...,19920.0,73200.0,030389
9,Port Royal Sales,95 FROEHLICH FARM BLVD WOODBURY NY 11797 USA,130886500,27889109,"PORT ROYAL SALES, LTD 95 FROEHLICH FARM BLVD W...",Tropical Food Mfg Ningbo,78 BINJIANG ZHILU XIAOGANG NINGBOCT:0574862285...,NaN,3919601,TROPICAL FOOD MANUFACTURING (NINGBO 78 BINJIAN...,19958.0,83200.0,160414


####

#### 5. (Cleaning) Condenses rows based on reference columns
#### Use reference IDs (ConsigneeLocalDUNS, ConsigneePanjivaID, ShipperPanjivaID)to identify rows that belong to the same underlying entity, even if names differ.

In [64]:
import pandas as pd
from pathlib import Path

#  Paths 
base_src = Path("/Users/nikhitha.khasnavis/Desktop/DataCleanse/input/us_imports_2015/std")
src_file = base_src / "us_import_2015_combined_std.csv"

base_out = Path("/Users/nikhitha.khasnavis/Desktop/DataCleanse/input/us_imports_2015/condense")
base_out.mkdir(parents=True, exist_ok=True)

out_df            = base_out / "us_import_2015_combined_std_combined.csv"
out_consig_report = base_out / "consignee_name_groups.csv"
out_ship_report   = base_out / "shipper_name_groups.csv"

df = pd.read_csv(src_file, dtype=str, low_memory=False)
n0 = len(df)

# Ensure reference and name columns exist and are clean strings
for c in ["ConsigneeLocalDUNS","ConsigneePanjivaID","ShipperPanjivaID",
          "ConsigneeName","ShipperName"]:
    if c not in df.columns:
        df[c] = ""
    else:
        df[c] = df[c].fillna("").astype(str).str.strip()

#  DSU (Union-Find) 
class DSU:
    """Disjoint Set Union for grouping by shared IDs."""
    def __init__(self, n): 
        self.p = list(range(n))
        self.r = [0]*n
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.r[ra] < self.r[rb]: self.p[ra] = rb
        elif self.r[ra] > self.r[rb]: self.p[rb] = ra
        else: self.p[rb] = ra; self.r[ra] += 1

def canonical_by_frequency(names):
    """Pick most frequent name; tie-break lexicographically."""
    s = pd.Series(names)
    vc = s.value_counts(dropna=False)
    top = vc.max()
    return sorted(vc[vc == top].index.tolist())[0]

#  Consignee grouping (by DUNS and/or PanjivaID) 
n = len(df)
by_duns, by_pid = {}, {}
for i,(duns,pid) in enumerate(zip(df["ConsigneeLocalDUNS"], df["ConsigneePanjivaID"])):
    if duns: by_duns.setdefault(duns,[]).append(i)
    if pid:  by_pid.setdefault(pid,[]).append(i)

cons_dsu = DSU(n)
for idxs in list(by_duns.values()) + list(by_pid.values()):
    for j in idxs[1:]:
        cons_dsu.union(idxs[0], j)

cons_comps = {}
for i in range(n):
    cons_comps.setdefault(cons_dsu.find(i), []).append(i)

cons_records = []
for idxs in cons_comps.values():
    names = [df.loc[i,"ConsigneeName"] for i in idxs]
    canonical = canonical_by_frequency(names)
    for i in idxs: 
        df.at[i,"ConsigneeName"] = canonical
    distinct_names = sorted(pd.unique(pd.Series(names)))
    cons_records.append({
        "distinct_names": len(distinct_names),
        "name_list": "; ".join(distinct_names),   # CSV-friendly
        "canonical_name": canonical,
        "consignee_duns": "; ".join(sorted({df.loc[i,"ConsigneeLocalDUNS"] for i in idxs if df.loc[i,"ConsigneeLocalDUNS"]})),
        "consignee_panjiva_ids": "; ".join(sorted({df.loc[i,"ConsigneePanjivaID"] for i in idxs if df.loc[i,"ConsigneePanjivaID"]})),
    })
cons_report = pd.DataFrame(cons_records)

#  Shipper grouping (by ShipperPanjivaID) 
by_ship = {}
for i, pid in enumerate(df["ShipperPanjivaID"]):
    if pid: by_ship.setdefault(pid,[]).append(i)

ship_dsu = DSU(n)
for idxs in by_ship.values():
    for j in idxs[1:]:
        ship_dsu.union(idxs[0], j)

ship_comps = {}
for i in range(n):
    ship_comps.setdefault(ship_dsu.find(i), []).append(i)

ship_records = []
for idxs in ship_comps.values():
    names = [df.loc[i,"ShipperName"] for i in idxs]
    canonical = canonical_by_frequency(names)
    for i in idxs: 
        df.at[i,"ShipperName"] = canonical
    distinct_names = sorted(pd.unique(pd.Series(names)))
    ship_records.append({
        "distinct_names": len(distinct_names),
        "name_list": "; ".join(distinct_names),   # CSV-friendly
        "canonical_name": canonical,
        "shipper_panjiva_ids": "; ".join(sorted({df.loc[i,"ShipperPanjivaID"] for i in idxs if df.loc[i,"ShipperPanjivaID"]})),
    })
ship_report = pd.DataFrame(ship_records)

df_out = df.drop_duplicates().reset_index(drop=True)

df_out.to_csv(out_df, index=False)
cons_report.to_csv(out_consig_report, index=False)
ship_report.to_csv(out_ship_report, index=False)

print(f"Saved dataset   : {out_df}")
print(f"Saved consignee : {out_consig_report}")
print(f"Saved shipper   : {out_ship_report}")
print(f"Input rows: {n0:,} -> Final rows: {len(df_out):,}")
print(f"Consignee groups: {len(cons_report):,}")
print(f"Shipper groups  : {len(ship_report):,}")

def _display_table(df_in: pd.DataFrame, cols, title: str, n: int = 10):
    """Slice BEFORE styling; hide index across pandas versions; wrap long text."""
    print(f"\n{title}")
    subset = df_in.loc[:, [c for c in cols if c in df_in.columns]].head(n)
    styler = subset.style.set_properties(**{"white-space": "pre-wrap"})
    # pandas >= 2.1: Styler.hide(axis="index"), older: hide_index()
    if hasattr(styler, "hide"):
        styler = styler.hide(axis="index")
    elif hasattr(styler, "hide_index"):
        styler = styler.hide_index()
    display(styler)

# Consignee: show reconciled groups, sorted descending
_cons_prev = cons_report.copy()
_cons_prev["distinct_names"] = pd.to_numeric(_cons_prev["distinct_names"], errors="coerce").fillna(0).astype(int)
if (_cons_prev["distinct_names"] > 1).any():
    _cons_prev = _cons_prev[_cons_prev["distinct_names"] > 1]
_cons_prev = _cons_prev.sort_values(["distinct_names","canonical_name"], ascending=[False, True]).copy()
_cons_prev["name_list"] = _cons_prev["name_list"].astype(str).str.replace("; ", "\n")  
_display_table(
    _cons_prev,
    ["distinct_names","canonical_name","name_list","consignee_duns","consignee_panjiva_ids"],
    title="Preview consignee groups",
    n=10
)

# Shipper: show first 10 rows 
_ship_prev = ship_report.copy()
_ship_prev["name_list"] = _ship_prev["name_list"].astype(str).str.replace("; ", "\n") 
_display_table(
    _ship_prev,
    ["distinct_names","canonical_name","name_list","shipper_panjiva_ids"],
    title="Preview shipper groups",
    n=10
)


Saved dataset   : /Users/nikhitha.khasnavis/Desktop/DataCleanse/input/us_imports_2015/condense/us_import_2015_combined_std_combined.csv
Saved consignee : /Users/nikhitha.khasnavis/Desktop/DataCleanse/input/us_imports_2015/condense/consignee_name_groups.csv
Saved shipper   : /Users/nikhitha.khasnavis/Desktop/DataCleanse/input/us_imports_2015/condense/shipper_name_groups.csv
Input rows: 101,504 -> Final rows: 77,749
Consignee groups: 11,653
Shipper groups  : 18,385

Preview consignee groups


distinct_names,canonical_name,name_list,consignee_duns,consignee_panjiva_ids
3,Bumble Bee Foods LLC,1 Bumble Bee Foods LLC Bumble Bags Bumble Bee Foods LLC,037713816; 135952609,27818233; 30475642; 44416235
3,Chicken Of The Sea Frozen Foods,Chicken Of The Sea Frozen Foods Thai Union Frozen Product Public Tri Union Frozen Products Incorporated,001706167,29039261; 44470413; 45984624
3,Ssc Incorporated Dba Sunnyvale Seafood,C O Ssc Incorporated Dba Sunnyvale Ssc Ssc Incorporated Dba Sunnyvale Seafood,183788991,27886120; 33502419; 44375598
3,The Great Fish Company,The Great Fish Company The Great Fish Company LLC Dba Riptide The Great Fish Holding Company LLC,079546935; 141242094,27827483; 34269242; 34729337
2,Aliments Altra Foods Incorporated,Aliments Altra Foods Incorporated Altra Food Distributors,205489961,29959902; 42244958
2,Aqua Star Usa Corporation,Aqua Star Aqua Star Usa Corporation,046704685,33483713; 46006229
2,Asian Commodities,Arko Food International Incorporated Asian Commodities,038672291,1861645; 45207474
2,Atalanta Corporation 1 Atalanta,Atalanta Corporation Atalanta Corporation 1 Atalanta,006980445,34454271; 44241071
2,B And I Overseas Trading Incorporated,B And I Overseas Trading Incorporated B And I Overseas Tranding Incorporated,118732275,27880759; 5169900
2,Bangkok Market,Bangkok Bangkok Imp And Exp Incorporated Bangkok Market,949005011,33463796; 44676260



Preview shipper groups


distinct_names,canonical_name,name_list,shipper_panjiva_ids
1,Zhanjiang Shuanghu Food,Zhanjiang Shuanghu Food,32579395
1,Qingdao Dencan Seafood Company,Qingdao Dencan Seafood Company,35036079
1,Xihe Food Company Limited,Xihe Food Company Limited,2167447
1,Zhongshan Foodstuffs And Aquatic,Zhongshan Foodstuffs And Aquatic,4069344
1,Oceanview Group Incorporated,Oceanview Group Incorporated,27792064
1,Qingdao Deli Trade Company Limited,Qingdao Deli Trade Company Limited,35552095
1,Guangxi Nanning Baiyang Food Company,Guangxi Nanning Baiyang Food Company,5096630
1,Dalian Meihe Foodstuff Company Limited,Dalian Meihe Foodstuff Company Limited,2284978
1,Fujian Yuehai Aquatic,Fujian Yuehai Aquatic,45866180
1,Tropical Food Mfg Ningbo,Tropical Food Mfg Ningbo,3919601


####

#### 6. (Cleaning) Clusters rows based on clustering logic
#### Clusters consignee / shipper names based on how similar they are (Levenshtein-like, 0.9). Canonical names within clusters are the most frequent names.

In [70]:
import re
import unicodedata
from pathlib import Path
from collections import defaultdict, Counter
from difflib import SequenceMatcher
import pandas as pd

#  Paths 
base = Path("/Users/nikhitha.khasnavis/Desktop/DataCleanse/input/us_imports_2015")
src  = base / "condense" / "us_import_2015_combined_std_combined.csv"  # <-- your input
out_dir = base / "cluster"
out_dir.mkdir(parents=True, exist_ok=True)

final_out     = out_dir / "final_combined_std_clustered.csv"
final_output  = out_dir / "final_output.csv"                # <-- renamed from corrections_names.csv
rep_cons_out  = out_dir / "consignee_clusters_report.csv"
rep_ship_out  = out_dir / "shipper_clusters_report.csv"

assert src.exists(), f"Input not found: {src}"

df = pd.read_csv(src, dtype=str, low_memory=False)
for c in ["ConsigneeName","ShipperName"]:
    if c in df.columns:
        df[c] = df[c].fillna("").astype(str).str.strip()
    else:
        df[c] = ""

# Pre-clean: drop a single leading "1" token (e.g., '1. Name' -> 'Name') 
LEADING_ONE_RE = re.compile(r'^\s*1(?=\s|[.\-_/])[\s.\-_/]*', flags=re.IGNORECASE)
def strip_leading_one(s: str) -> str:
    if not isinstance(s, str):
        s = "" if s is None else str(s)
    return LEADING_ONE_RE.sub("", s).strip()

df["ConsigneeName"] = df["ConsigneeName"].apply(strip_leading_one)
df["ShipperName"]   = df["ShipperName"].apply(strip_leading_one)

#  Value / Quantity columns (pick first present) 
VALUE_CANDS = ["ValueOfGoodsUSD","ValueOfGoodsFOBUSD","ExportValue","InvoiceValueUSD",
               "ValueUSD","Value_USD","FOBUSD","CIFUSD","Value"]
QTY_CANDS   = ["GrossWeightKg","NetWeightKg","WeightKg","Weightkg","Quantity (kg)","Quantity_kg","QuantityKg"]

def pick_first_existing(cols, cands):
    for k in cands:
        if k in cols:
            return k
    return None

VAL_COL = pick_first_existing(df.columns, VALUE_CANDS)
QTY_COL = pick_first_existing(df.columns, QTY_CANDS)

def as_float(x):
    """Safe float (strip commas/$, missing -> 0.0)."""
    try:
        return float(str(x).replace(",","").replace("$",""))
    except Exception:
        return 0.0

val_series = df[VAL_COL].map(as_float) if VAL_COL else pd.Series([0.0]*len(df))
qty_series = df[QTY_COL].map(as_float) if QTY_COL else pd.Series([0.0]*len(df))

#  Normalization helper for similarity
PUNCT_TO_SPACE = re.compile(r"[\,\.;:\|\(\)\[\]\{\}/\\\+\=\*\!\?\#\^\"“”‘’`~_–—\-]+")
NUM_TOKEN      = re.compile(r"\b\d+\b")

# Remove generic legal suffixes ONLY FOR MATCHING (not for writing)
# 1) composites like "S. de R.L. de C.V.", "S.A. de C.V." removed as a block
COMPOSITES_TO_STRIP = [
    re.compile(r"\bs\.?\s*de\s*r\.?\s*l\.?\s*de\s*c\.?\s*v\.?\b", re.I),
    re.compile(r"\bs\.?\s*a\.?\s*de\s*c\.?\s*v\.?\b", re.I),
    re.compile(r"\bs\.?\s*de\s*r\.?\s*l\.?\b", re.I),
]

# 2) single tokens to drop from matching
LEGAL_STOPWORDS = {
    "inc","incorporated","corp","corporation","co","company",
    "ltd","limited","ltda","pty","plc","llc","llp","gmbh","ag","bv","nv",
    "sa","srl","spa","pte","ptyltd","group","intl","international","int",
    "holdings","holding"
}

def norm_for_match(x: str, strip_numbers: bool = False) -> str:
    """
    Normalize for similarity: lowercase, ASCII fold, remove punctuation.
    Then strip legal/composite terms that can cause false merges.
    Optionally strip standalone numbers.
    """
    if x is None:
        return ""
    s = unicodedata.normalize("NFKD", str(x)).encode("ascii","ignore").decode("ascii").lower()
    s = s.replace("&"," and ")
    s = PUNCT_TO_SPACE.sub(" ", s)
    s = re.sub(r"\s+"," ", s).strip()

    # strip composites
    for pat in COMPOSITES_TO_STRIP:
        s = pat.sub(" ", s)

    # strip legal stopwords token-by-token
    toks = [t for t in s.split() if t not in LEGAL_STOPWORDS]
    s = " ".join(toks)

    if strip_numbers:
        s = NUM_TOKEN.sub("", s)
        s = re.sub(r"\s+"," ", s).strip()

    return s

def length_bucket(s: str, size:int=3) -> int:
    """Rough length bucket to reduce comparisons."""
    return max(0, len(s)//size)

# Clustering with numeric-aware comparison 
CUTOFF = 0.90

def cluster_names_by_similarity(names, counts, cutoff=CUTOFF):
    """Cluster names by similarity, comparing both normal and number-stripped forms."""
    recs = []
    for nm in names:
        n1 = norm_for_match(nm, strip_numbers=False)
        n2 = norm_for_match(nm, strip_numbers=True)
        key = (n1[:1], length_bucket(n1))
        recs.append((nm, n1, n2, key))

    # Block by (first char, length bucket)
    blocks = defaultdict(list)
    for nm, n1, n2, key in recs:
        blocks[key].append((nm, n1, n2))

    clusters = []
    for key, items in blocks.items():
        if len(items) == 1:
            clusters.append([items[0][0]])
            continue

        # Union-Find within block
        parent = list(range(len(items)))
        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x
        def union(a,b):
            ra, rb = find(a), find(b)
            if ra == rb: return
            parent[rb] = ra

        # Pairwise compare within block
        for i in range(len(items)):
            for j in range(i+1, len(items)):
                a, a1, a2 = items[i]
                b, b1, b2 = items[j]
                if abs(len(a1)-len(b1)) > 6:
                    continue
                r_norm = SequenceMatcher(None, a1, b1).ratio()
                r_num  = SequenceMatcher(None, a2, b2).ratio()
                if max(r_norm, r_num) >= cutoff:
                    union(i, j)

        # Collect connected components
        comp = defaultdict(list)
        for idx in range(len(items)):
            comp[find(idx)].append(items[idx][0])
        clusters.extend(comp.values())

    return clusters

def canonical_name(cluster, counts):
    """Pick most frequent as canonical; tie-break lexicographically."""
    vc = Counter({nm: counts.get(nm, 0) for nm in cluster})
    top = max(vc.values())
    return sorted([k for k,v in vc.items() if v == top])[0]

fmt1 = lambda x: f"{x:,.1f}"  # "108,000.0"

def build_for(name_col: str, dataset_label="US Imports 2015"):
    """Cluster one column and build report + corrections + after-series."""
    counts = df[name_col].value_counts()
    unique = list(counts.index)
    clusters = cluster_names_by_similarity(unique, counts.to_dict(), cutoff=CUTOFF)

    name2canon = {}
    rep_rows = []
    for cl in clusters:
        canon = canonical_name(cl, counts.to_dict())
        for nm in cl:
            name2canon[nm] = canon
        rep_rows.append({
            "distinct_names": len(sorted(set(cl))),
            "name_list": "; ".join(sorted(set(cl))),  # CSV-friendly
            "canonical_name": canon,
        })
    report_df = pd.DataFrame(rep_rows).sort_values(
        ["distinct_names","canonical_name"], ascending=[False, True]
    ).reset_index(drop=True)

    agg = pd.DataFrame({
        "before": df[name_col],
        "_val": val_series,
        "_qty": qty_series
    }).groupby("before", dropna=False).agg(
        frequency=("before","size"),
        value_usd=("_val","sum"),
        quantity_kg=("_qty","sum")
    ).reset_index()
    agg["after"] = agg["before"].map(name2canon).fillna(agg["before"])
    agg["Value_USD (of before)"] = agg["value_usd"].map(fmt1)
    agg["Quantity (kg)"]         = agg["quantity_kg"].map(fmt1)

    col_label = "Consignee" if name_col == "ConsigneeName" else "Shipper"
    corr = agg[["before","after","frequency","Value_USD (of before)","Quantity (kg)"]].copy()
    corr.insert(0,"Column Name", col_label)
    corr.insert(0,"Dataset", dataset_label)
    corr["ManualCorrection"] = ""
    corr["Reason (-1,0,1)"]  = ""
    corr["Canonical"]        = ""
    corr = corr[["Dataset","Column Name","before","after","frequency",
                 "Value_USD (of before)","Quantity (kg)",
                 "ManualCorrection","Reason (-1,0,1)","Canonical"]]

    after_series = df[name_col].map(name2canon).fillna(df[name_col])
    return corr, report_df, after_series

# Build for both columns 
corr_cons, rep_cons, after_cons = build_for("ConsigneeName")
corr_ship, rep_ship, after_ship = build_for("ShipperName")

# Combine corrections into ONE output (renamed to final_output.csv)
final_output_df = pd.concat([corr_cons, corr_ship], ignore_index=True)

# Apply canonical names and remove exact dup rows
df_out = df.copy()
df_out["ConsigneeName"] = after_cons
df_out["ShipperName"]   = after_ship
before_rows = len(df_out)
df_out = df_out.drop_duplicates().reset_index(drop=True)
removed = before_rows - len(df_out)

# Save to disk
final_output_df.to_csv(final_output, index=False)
rep_cons.to_csv(rep_cons_out, index=False)
rep_ship.to_csv(rep_ship_out, index=False)
df_out.to_csv(final_out, index=False)

print(f"Files written to: {out_dir}")
print(f"Cutoff used: {CUTOFF:.2f}")

# ---- PREVIEWS (show every name on its own line) ----
def _display_wrapped(df_in: pd.DataFrame, cols, title: str, n: int = 10):
    print(f"\n{title}")
    subset = df_in.loc[:, [c for c in cols if c in df_in.columns]].head(n)
    with pd.option_context('display.max_colwidth', None):
        subset = subset.copy()
        if "name_list" in subset.columns:
            subset["name_list"] = subset["name_list"].astype(str).str.replace("; ", "\n")
        styler = subset.style.set_properties(**{"white-space": "pre-wrap"})
        if hasattr(styler, "hide"):
            styler = styler.hide(axis="index")
        elif hasattr(styler, "hide_index"):
            styler = styler.hide_index()
        display(styler)

print("\nPreview — Consignee clusters report:")
rep_cons_sorted = rep_cons.sort_values(["distinct_names","canonical_name"], ascending=[False, True])
_display_wrapped(rep_cons_sorted, ["distinct_names","canonical_name","name_list"], "Consignee clusters", 10)

print("\nPreview — Shipper clusters report:")
_display_wrapped(rep_ship, ["distinct_names","canonical_name","name_list"], "Shipper clusters", 10)

print("\nPreview — Final output:")
_display_wrapped(final_output_df, ["Dataset","Column Name","before","after","frequency","Value_USD (of before)","Quantity (kg)"], "Final output", 12)


Files written to: /Users/nikhitha.khasnavis/Desktop/DataCleanse/input/us_imports_2015/cluster
Cutoff used: 0.90

Preview — Consignee clusters report:

Consignee clusters


distinct_names,canonical_name,name_list
5,Vandegrift Forwarding Company Incorporated,Vandegrift Forwarding Vandegrift Forwarding Company Vandegrift Forwarding Company Incorporated Vandergrift Forwarding Vandergrift Forwarding Company Incorporated
4,Flegenheimer International Incorporated,Flegebheimer International Flegenhaimer Flegenheimer International Flegenheimer International Incorporated
3,Anh Quan International Trading,Anh Quan International Tradig Company Anh Quan International Trading Anh Quan International Trading Company
3,Dalian Rifu Food Company Limited,Dalian Rifu Food Company Limited Dalian Taifu Food Dalian Taifu Food Company Limited
3,G And L Seafood,G And L Sea Foods Incorporated G And L Seafood G And L Seafood Incorporated
3,Gourmet International Incorporated,Gourme International Limited Gourmet International Gourmet International Incorporated
3,International Marine Industries Incorporated,International Marine Industries Incorporated Intl Marine Industrie Intl Marine Industries Incorporated
3,Osborne,Osborne Osbourne Osbourne Limited
3,Premium Seafood Company Incorporated,Premium Seafood Company Incorporated Premium Seafood Limited Premium Seafoods Limited
3,Shorlinez Incorporated,Shoreline International Shorelinez Shorlinez Incorporated



Preview — Shipper clusters report:

Shipper clusters


distinct_names,canonical_name,name_list
3,Alliance Select Foods,Alliance Select Foods Alliance Select Foods International Alliance Select Foods International Incorporated
3,Asian Alliance International Company Limited,Asian Alliance Asian Alliance International Asian Alliance International Company Limited
3,Dalian Hongxu Food Company Limited,Dalian Hongfu Food Company Limited Dalian Honghui Food Company Limited Dalian Hongxu Food Company Limited
3,Exportadora de Productos Del Oceano,Exportador de Productos Del Oceano Exportadora de Productos Del Ocea Exportadora de Productos Del Oceano
3,Hee Chang Trading Company Limited,Hee Chang Rading Company Limited Hee Chang Trading Company Limited Heechang Trading Company Limited
3,Thai Union Group,Thai Union Thai Union Group Thai Union International Incorporated
2,Acadian Fishermens Company Op Assoc,Acadian Fishermen S Company Op Ass N Limited Acadian Fishermens Company Op Assoc
2,Alimentos Sumar,Alimentos Sumar Alimentos Sumar SA
2,Allied Pacific Food Dalian Company,Allied Pacific Food Dalian Company Allied Pacific Food Dalian Company Limited
2,Apex Frozen Foods,A P Frozen Foods Company Limited Apex Frozen Foods



Preview — Final output:

Final output


Dataset,Column Name,before,after,frequency,Value_USD (of before),Quantity (kg)
US Imports 2015,Consignee,,,3769,"424,855,898.0","66,528,670.2"
US Imports 2015,Consignee,0 F,0 F,1,"108,000.0","18,975.0"
US Imports 2015,Consignee,0939578 B C Limited,0939578 B C Limited,2,"167,100.0","47,025.0"
US Imports 2015,Consignee,1000 Corporate Center Dr Ste 120,1000 Corporate Center Dr Ste 120,1,"116,000.0","20,250.0"
US Imports 2015,Consignee,101 Lucas Valley Road Suite 307,101 Lucas Valley Road Suite 307,1,"108,000.0","18,500.0"
US Imports 2015,Consignee,11 15 Parker Street,11 15 Parker Street,1,"189,000.0","18,500.0"
US Imports 2015,Consignee,120008 Tesoros Trading Company,120008 Tesoros Trading Company,5,"120,000.0","74,414.0"
US Imports 2015,Consignee,138 Lomita St El Segundo Ca 90245 T,138 Lomita St El Segundo Ca 90245 T,3,"162,900.0","31,308.0"
US Imports 2015,Consignee,1750 112 Th Ave Ne Suite C236,1750 112 Th Ave Ne Suite C236,5,"674,000.0","113,720.0"
US Imports 2015,Consignee,2 Nd Jamie Helterman,2 Nd Jamie Helterman,2,"183,500.0","41,280.0"
